In [ ]:
# ==============================================================================
# BAGIAN 1 — IMPORT LIBRARY
# ==============================================================================

import os
import cv2
import glob
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import nibabel as nib
import kagglehub

from datetime import datetime
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import LabelBinarizer
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {tf.config.list_physical_devices('GPU')}")
print()

In [ ]:
# ==============================================================================
# BAGIAN 1B — DETERMINISME PENUH (agar hasil run konsisten antar sesi)
# ==============================================================================
# Tanpa baris ini, operasi GPU (cuDNN) TensorFlow bersifat non-deterministik
# secara default walau SEED sudah di-set, sehingga akurasi bisa berbeda-beda
# tiap kali notebook dijalankan ulang meski kode identik. Baris ini memaksa
# TensorFlow memakai algoritma deterministik di setiap operasi.

os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['PYTHONHASHSEED'] = str(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("[OK] Mode deterministik TensorFlow diaktifkan.")
except Exception as e:
    print(f"[PERINGATAN] Tidak bisa mengaktifkan determinisme penuh: {e}")

print("Catatan: hasil training tetap bisa sedikit berbeda antar jenis GPU/driver "
      "CUDA yang berbeda, namun pada mesin & environment yang sama, hasil "
      "SEHARUSNYA identik tiap kali dijalankan ulang.")

In [ ]:
# ==============================================================================
# BAGIAN 2 — KONFIGURASI GLOBAL
# ==============================================================================

IMG_SIZE   = 128
CHANNELS   = 3
BATCH_SIZE = 32
OUTPUT_DIR = '/kaggle/working/output_hybrid'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSES     = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = len(CLASSES)

# Konfigurasi per stage
EPOCHS_AE       = 50     # Stage 1: Autoencoder
LR_AE           = 1e-3
EPOCHS_CLF      = 30     # Stage 2: Classifier (encoder frozen)
LR_CLF          = 1e-3
EPOCHS_FINETUNE = 20     # Stage 3: Fine-tuning
LR_FINETUNE     = 1e-5

print(f"IMG_SIZE={IMG_SIZE} | BATCH={BATCH_SIZE}")
print(f"Stage1 AE   : epochs={EPOCHS_AE},  lr={LR_AE}")
print(f"Stage2 CLF  : epochs={EPOCHS_CLF}, lr={LR_CLF}")
print(f"Stage3 Fine : epochs={EPOCHS_FINETUNE}, lr={LR_FINETUNE}")
print(f"Output: {OUTPUT_DIR}")
print()

In [ ]:
# ==============================================================================
# BAGIAN 3 — DOWNLOAD DATASET
# ==============================================================================

print("=" * 65)
print("  MENGUNDUH DATASET DARI KAGGLE")
print("=" * 65)

# Dataset 1: Dataset utama (Masoud Nickparvar) — 4 kelas, struktur Training/Testing
print("[1/3] Mengunduh brain-tumor-mri-dataset...")
PATH_MAIN  = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print(f"      → {PATH_MAIN}")

# Dataset 2: Dataset tambahan heterogen (ishans24)
print("[2/3] Mengunduh brain-tumor-dataset (ishans24)...")
PATH_EXTRA = kagglehub.dataset_download("ishans24/brain-tumor-dataset")
print(f"      → {PATH_EXTRA}")

# Dataset 3: BraTS 2020 — format NIfTI (.nii.gz)
print("[3/3] Mengunduh BraTS20 dataset...")
PATH_BRATS = kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")
print(f"      → {PATH_BRATS}")

print("\nSemua dataset berhasil diunduh.\n")

In [ ]:
# ==============================================================================
# BAGIAN 3A-VIS — VISUALISASI RINGKASAN DATASET SETELAH DIUNDUH
# ==============================================================================
# Menampilkan jumlah file per sumber dataset persis setelah proses download,
# sebelum masuk ke tahap pembersihan data.

def count_files_by_ext(root, exts=('.jpg', '.jpeg', '.png', '.bmp')):
    n = 0
    for r, d, files in os.walk(root):
        for f in files:
            if f.lower().endswith(exts):
                n += 1
    return n

def count_nii_files(root):
    n = 0
    for r, d, files in os.walk(root):
        for f in files:
            if f.endswith('.nii') or f.endswith('.nii.gz'):
                n += 1
    return n

_n_main_dl  = count_files_by_ext(PATH_MAIN)
_n_extra_dl = count_files_by_ext(PATH_EXTRA)
_n_brats_dl = count_nii_files(PATH_BRATS)

print("=" * 65)
print("  RINGKASAN DATASET SETELAH DIUNDUH (SEBELUM CLEANING)")
print("=" * 65)
print(f"  Dataset Utama (MAIN)   : {_n_main_dl} file gambar")
print(f"  Dataset Tambahan (EXTRA): {_n_extra_dl} file gambar")
print(f"  BraTS 2020 (NIfTI)     : {_n_brats_dl} file .nii/.nii.gz")

fig, ax = plt.subplots(figsize=(7, 4.5))
_labels_dl = ['MAIN\n(brain-tumor-mri)', 'EXTRA\n(ishans24)', 'BraTS 2020\n(.nii.gz)']
_counts_dl = [_n_main_dl, _n_extra_dl, _n_brats_dl]
_colors_dl = ['#3498db', '#9b59b6', '#e67e22']
bars = ax.bar(_labels_dl, _counts_dl, color=_colors_dl)
for b, c in zip(bars, _counts_dl):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + max(_counts_dl)*0.01,
            str(c), ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Jumlah File')
ax.set_title('Jumlah File per Sumber Dataset Setelah Diunduh (Sebelum Cleaning)',
              fontweight='bold')
plt.tight_layout()
_p_dl = os.path.join(OUTPUT_DIR, 'ringkasan_dataset_setelah_download.png')
plt.savefig(_p_dl, dpi=130, bbox_inches='tight')
plt.show()
print(f"[DISIMPAN] {_p_dl}")

In [ ]:
# ==============================================================================
# BAGIAN 3B — PEMBERSIHAN DATA: HAPUS GAMBAR CORRUPT & BERUKURAN 0 BYTE
# ==============================================================================

import shutil

VALID_EXT  = ('.jpg', '.jpeg', '.png', '.bmp')
CLEAN_ROOT = os.path.join(OUTPUT_DIR, '..', 'dataset_clean')
CLEAN_ROOT = os.path.abspath(CLEAN_ROOT)
os.makedirs(CLEAN_ROOT, exist_ok=True)

def clean_dataset_copy(src_root: str, dst_root: str, source_name: str = "Dataset"):
    print(f"\n[{source_name}] Membersihkan: {src_root}")
    print(f"           -> Output bersih: {dst_root}")

    if os.path.exists(dst_root):
        shutil.rmtree(dst_root)

    # FIX REPRODUCIBILITY sorted() agar urutan file konsisten antar run
    all_files = []
    for root, dirs, files in os.walk(src_root):
        dirs.sort()
        for fname in sorted(files):
            if fname.lower().endswith(VALID_EXT):
                all_files.append(os.path.join(root, fname))
    all_files = sorted(all_files)

    n_zero, n_corrupt, n_ok = 0, 0, 0

    for fpath in tqdm(all_files, desc=f"Cek {source_name}"):
        try:
            # 1) Cek ukuran file 0 byte
            if os.path.getsize(fpath) == 0:
                n_zero += 1
                continue

            # 2) Cek apakah file bisa didekode sebagai gambar valid
            data = np.fromfile(fpath, dtype=np.uint8)
            img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
            if img is None or img.size == 0:
                n_corrupt += 1
                continue

            # 3) Valid -> salin ke direktori bersih, pertahankan struktur folder
            rel_path = os.path.relpath(fpath, src_root)
            dst_path = os.path.join(dst_root, rel_path)
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy2(fpath, dst_path)
            n_ok += 1

        except Exception:
            n_corrupt += 1

    print(f"  Total gambar discan : {len(all_files)}")
    print(f"  Dilewati (0 byte)   : {n_zero}")
    print(f"  Dilewati (corrupt)  : {n_corrupt}")
    print(f"  Disalin (valid)     : {n_ok}")

    return n_zero, n_corrupt, len(all_files), dst_root

print("=" * 65)
print("  PEMBERSIHAN DATA: SCAN & SALIN HANYA GAMBAR VALID")
print("=" * 65)

CLEAN_MAIN  = os.path.join(CLEAN_ROOT, 'main')
CLEAN_EXTRA = os.path.join(CLEAN_ROOT, 'extra')

z1, c1, t1, CLEAN_MAIN  = clean_dataset_copy(PATH_MAIN,  CLEAN_MAIN,  "MAIN")
z2, c2, t2, CLEAN_EXTRA = clean_dataset_copy(PATH_EXTRA, CLEAN_EXTRA, "EXTRA")

# Mulai dari titik ini, gunakan PATH_MAIN/PATH_EXTRA versi bersih
PATH_MAIN  = CLEAN_MAIN
PATH_EXTRA = CLEAN_EXTRA

print("\nRINGKASAN PEMBERSIHAN")
print(f"  Total discan        : {t1 + t2}")
print(f"  Total dilewati      : {z1 + c1 + z2 + c2} "
      f"(0 byte: {z1 + z2}, corrupt: {c1 + c2})")
print(f"Dataset bersih tersedia di:")
print(f"     PATH_MAIN  = {PATH_MAIN}")
print(f"     PATH_EXTRA = {PATH_EXTRA}")

# VISUAL Ringkasan jumlah data SEBELUM vs SESUDAH penghapusan corrupt/0-byte
_labels_3b   = ['MAIN', 'EXTRA']
_before_3b   = [t1, t2]
_dihapus_3b  = [z1 + c1, z2 + c2]
_sesudah_3b  = [t1 - (z1 + c1), t2 - (z2 + c2)]

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(_labels_3b))
width = 0.35
ax.bar(x - width/2, _before_3b, width, label='Sebelum (Total Scan)', color='#95a5a6')
ax.bar(x + width/2, _sesudah_3b, width, label='Sesudah (Valid)', color='#2ecc71')
ax.set_xticks(x)
ax.set_xticklabels(_labels_3b)
ax.set_ylabel('Jumlah Gambar')
ax.set_title('Perbandingan Jumlah Data Sebelum vs Sesudah Hapus Corrupt/0-byte',
              fontweight='bold')
ax.legend()
for i, (b, s) in enumerate(zip(_before_3b, _sesudah_3b)):
    ax.text(i - width/2, b + 5, str(b), ha='center', fontsize=9)
    ax.text(i + width/2, s + 5, str(s), ha='center', fontsize=9)
plt.tight_layout()
_p3b = os.path.join(OUTPUT_DIR, 'perbandingan_cleaning_corrupt.png')
plt.savefig(_p3b, dpi=130, bbox_inches='tight')
plt.show()
print(f"[DISIMPAN] {_p3b}")

In [ ]:
# ==============================================================================
# BAGIAN 3C — DATA CLEANING: HAPUS GAMBAR DUPLIKAT (IMAGE HASHING)
# ==============================================================================

HASH_SIZE = 8  # menghasilkan hash 64-bit

def compute_ahash(fpath: str, hash_size: int = HASH_SIZE):
    """Hitung average hash (aHash) dari sebuah file gambar. Return: int hash atau None jika gagal."""
    try:
        data = np.fromfile(fpath, dtype=np.uint8)
        img  = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        small = cv2.resize(img, (hash_size, hash_size), interpolation=cv2.INTER_AREA)
        avg   = small.mean()
        bits  = (small > avg).flatten()
        h = 0
        for bit in bits:
            h = (h << 1) | int(bit)
        return h
    except Exception:
        return None

def collect_image_files(base_path: str):
    files = []
    for root, dirs, fnames in os.walk(base_path):
        dirs.sort()  # urutkan subdirektori agar traversal konsisten
        for fname in sorted(fnames):
            if fname.lower().endswith(VALID_EXT):
                files.append(os.path.join(root, fname))
    return sorted(files)

def deduplicate_paths(path_groups: dict):
    seen_hashes = {}
    n_dup_per_source = {src: 0 for src in path_groups}

    print("=" * 65)
    print("  DATA CLEANING: DETEKSI & HAPUS GAMBAR DUPLIKAT (IMAGE HASHING)")
    print("=" * 65)

    for src, base_path in path_groups.items():
        files = collect_image_files(base_path)
        for fpath in tqdm(files, desc=f"Hashing {src}"):
            h = compute_ahash(fpath)
            if h is None:
                continue  # gagal dihash -> dipertahankan, ditangani tahap lain

            if h in seen_hashes:
                try:
                    os.remove(fpath)
                    n_dup_per_source[src] += 1
                except OSError:
                    pass
            else:
                seen_hashes[h] = fpath

        print(f"  [{src}] Duplikat dihapus: {n_dup_per_source[src]} "
              f"(sisa: {len(files) - n_dup_per_source[src]} dari {len(files)})")

    print(f"\nTotal duplikat dihapus: {sum(n_dup_per_source.values())}")
    print("Deduplikasi selesai. Risiko data leakage antar dataset diminimalkan.")
    return n_dup_per_source

_dup_counts = deduplicate_paths({
    "MAIN":  PATH_MAIN,
    "EXTRA": PATH_EXTRA,
})

# VISUAL Ringkasan jumlah data SEBELUM vs SESUDAH deduplikasi
_labels_3c = list(_dup_counts.keys())
_removed_3c = list(_dup_counts.values())
_before_3c = [len(collect_image_files(p)) + r for p, r in\
              zip([PATH_MAIN, PATH_EXTRA], _removed_3c)]
_after_3c  = [b - r for b, r in zip(_before_3c, _removed_3c)]

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(_labels_3c))
width = 0.35
ax.bar(x - width/2, _before_3c, width, label='Sebelum Deduplikasi', color='#95a5a6')
ax.bar(x + width/2, _after_3c, width, label='Sesudah Deduplikasi', color='#3498db')
ax.set_xticks(x)
ax.set_xticklabels(_labels_3c)
ax.set_ylabel('Jumlah Gambar')
ax.set_title('Perbandingan Jumlah Data Sebelum vs Sesudah Hapus Duplikat',
              fontweight='bold')
ax.legend()
for i, (b, a) in enumerate(zip(_before_3c, _after_3c)):
    ax.text(i - width/2, b + 20, str(b), ha='center', fontsize=9)
    ax.text(i + width/2, a + 20, str(a), ha='center', fontsize=9)
plt.tight_layout()
_p3c = os.path.join(OUTPUT_DIR, 'perbandingan_cleaning_duplikat.png')
plt.savefig(_p3c, dpi=130, bbox_inches='tight')
plt.show()
print(f"[DISIMPAN] {_p3c}")

In [ ]:
# ==============================================================================
# BAGIAN 3D — DETEKSI OUTLIER / DATA MENYIMPANG
# ==============================================================================
def collect_image_stats(base_path: str, source_name: str = "Dataset"):
    """Kumpulkan statistik (dimensi, brightness, kontras) untuk setiap gambar."""
    files = collect_image_files(base_path)
    rows = []

    for fpath in tqdm(files, desc=f"Analisis statistik {source_name}"):
        try:
            data = np.fromfile(fpath, dtype=np.uint8)
            img_gray = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
            if img_gray is None:
                continue
            h, w = img_gray.shape[:2]
            rows.append({
                "path": fpath,
                "source": source_name,
                "width": w,
                "height": h,
                "aspect_ratio": round(w / h, 3),
                "brightness_mean": float(img_gray.mean()),
                "contrast_std": float(img_gray.std()),
            })
        except Exception:
            continue

    return pd.DataFrame(rows)

def flag_outliers_iqr(df: pd.DataFrame, columns: list):
    """Tandai baris outlier berdasarkan metode IQR untuk setiap kolom numerik."""
    df = df.copy()
    df["is_outlier"] = False
    outlier_reasons = {col: 0 for col in columns}

    for col in columns:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask = (df[col] < lower) | (df[col] > upper)
        df.loc[mask, "is_outlier"] = True
        outlier_reasons[col] = int(mask.sum())

    return df, outlier_reasons

print("=" * 65)
print("  DETEKSI OUTLIER / DATA MENYIMPANG")
print("=" * 65)

df_stats_main  = collect_image_stats(PATH_MAIN,  "MAIN")
df_stats_extra = collect_image_stats(PATH_EXTRA, "EXTRA")
df_stats_all   = pd.concat([df_stats_main, df_stats_extra], ignore_index=True)

METRIC_COLS = ["width", "height", "aspect_ratio", "brightness_mean", "contrast_std"]
df_stats_all, outlier_reasons = flag_outliers_iqr(df_stats_all, METRIC_COLS)

n_outlier = int(df_stats_all["is_outlier"].sum())
n_total   = len(df_stats_all)

print(f"\nRINGKASAN OUTLIER {n_outlier} dari {n_total} gambar "
      f"({100*n_outlier/n_total:.2f}%) terindikasi outlier pada minimal 1 metrik.")
print("\nJumlah outlier per metrik:")
for col, cnt in outlier_reasons.items():
    print(f"  {col:18s}: {cnt} gambar")

# Statistik deskriptif keseluruhan (untuk naskah skripsi)
print("\nSTATISTIK DESKRIPTIF DATASET")
print(df_stats_all[METRIC_COLS].describe().round(3).to_string())

# Visualisasi distribusi metrik (boxplot) untuk identifikasi outlier
fig, axes = plt.subplots(1, len(METRIC_COLS), figsize=(4 * len(METRIC_COLS), 4))
fig.suptitle("Distribusi Metrik Citra & Deteksi Outlier (Boxplot)", fontweight='bold')

for ax, col in zip(axes, METRIC_COLS):
    sns.boxplot(y=df_stats_all[col], ax=ax, color='#3498db')
    ax.set_title(col, fontsize=10)
    ax.set_ylabel("")

plt.tight_layout()
_p = os.path.join(OUTPUT_DIR, 'outlier_boxplot.png')
plt.savefig(_p, dpi=130, bbox_inches='tight')
plt.show()
print(f"[DISIMPAN] {_p}")

# Tampilkan contoh visual gambar yang terdeteksi outlier (jika ada)
df_outliers = df_stats_all[df_stats_all["is_outlier"]].copy()

if len(df_outliers) > 0:
    n_show = min(10, len(df_outliers))
    sample_outliers = df_outliers.sample(n_show, random_state=SEED)

    fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2.2, 2.6))
    if n_show == 1:
        axes = [axes]
    fig.suptitle(f"Contoh Gambar Terdeteksi Outlier ({n_show} dari {len(df_outliers)})",
                 fontweight='bold')

    for ax, (_, row) in zip(axes, sample_outliers.iterrows()):
        data = np.fromfile(row["path"], dtype=np.uint8)
        img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{row['width']}x{row['height']}\nbr={row['brightness_mean']:.0f}",
                     fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    _p2 = os.path.join(OUTPUT_DIR, 'outlier_examples.png')
    plt.savefig(_p2, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {_p2}")
else:
    print("\nTidak ditemukan outlier signifikan pada dataset.")

# Simpan detail outlier ke CSV untuk lampiran/audit lebih lanjut
_csv_path = os.path.join(OUTPUT_DIR, 'outlier_report.csv')
df_outliers.to_csv(_csv_path, index=False)
print(f"\n[DISIMPAN] Detail outlier: {_csv_path}")

In [ ]:
# ==============================================================================
# BAGIAN 3E — AUDIT VISUAL BERTARGET (OUTLIER PALING EKSTREM & DETEKSI WATERMARK)
# ==============================================================================

# 1) AUDIT EKSTREM: 8 gambar paling ekstrem untuk setiap metrik
def show_extreme_samples(df: pd.DataFrame, metric: str, n: int = 8):
    """Tampilkan n gambar dengan nilai TERTINGGI dan n gambar TERENDAH pada metric."""
    df_sorted = df.sort_values(metric)
    lowest  = df_sorted.head(n)
    highest = df_sorted.tail(n)

    fig, axes = plt.subplots(2, n, figsize=(n * 2.0, 4.6))
    fig.suptitle(f"Audit Ekstrem: {metric} (Terendah vs Tertinggi)",
                 fontsize=13, fontweight='bold')

    for row_idx, (group, label) in enumerate([(lowest, "Terendah"), (highest, "Tertinggi")]):
        for col_idx, (_, row) in enumerate(group.iterrows()):
            data = np.fromfile(row["path"], dtype=np.uint8)
            img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
            img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            ax = axes[row_idx, col_idx]
            ax.imshow(img)
            ax.set_title(f"{row[metric]:.1f}", fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            if col_idx == 0:
                ax.set_ylabel(label, fontsize=10, fontweight='bold')

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, f'audit_ekstrem_{metric}.png')
    plt.savefig(p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p}")

print("=" * 65)
print("  AUDIT VISUAL BERTARGET: SAMPEL PALING EKSTREM PER METRIK")
print("=" * 65)

for metric in ["brightness_mean", "contrast_std"]:
    show_extreme_samples(df_stats_all, metric, n=8)

# 2) HEURISTIK DETEKSI WATERMARK / TEKS DI SUDUT GAMBAR
def detect_watermark_corner(img_gray: np.ndarray, corner_frac: float = 0.18,
                             white_thresh: int = 235, min_white_ratio: float = 0.03):
    """
    Cek 4 sudut gambar untuk indikasi watermark/teks putih.
    Return: (terdeteksi: bool, sudut_terdampak: list[str])
    """
    h, w = img_gray.shape[:2]
    ch, cw = int(h * corner_frac), int(w * corner_frac)

    corners = {
        "kiri-atas":  img_gray[0:ch, 0:cw],
        "kanan-atas": img_gray[0:ch, w-cw:w],
        "kiri-bawah": img_gray[h-ch:h, 0:cw],
        "kanan-bawah":img_gray[h-ch:h, w-cw:w],
    }

    flagged = []
    for name, patch in corners.items():
        white_ratio = np.mean(patch > white_thresh)
        local_std   = patch.std()
        # Watermark: proporsi piksel putih murni cukup tinggi DAN ada variasi
        # lokal (bukan area putih polos besar seperti latar MRI kosong)
        if white_ratio > min_white_ratio and local_std > 40:
            flagged.append(name)

    return len(flagged) > 0, flagged

print("\n" + "=" * 65)
print("  HEURISTIK DETEKSI WATERMARK / TEKS DI SUDUT GAMBAR")
print("=" * 65)

watermark_candidates = []
for _, row in tqdm(df_stats_all.iterrows(), total=len(df_stats_all),
                    desc="Scan watermark"):
    data = np.fromfile(row["path"], dtype=np.uint8)
    img_gray = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        continue
    is_flagged, corners_hit = detect_watermark_corner(img_gray)
    if is_flagged:
        watermark_candidates.append({
            "path": row["path"],
            "source": row["source"],
            "sudut_terdampak": ", ".join(corners_hit),
        })

df_watermark = pd.DataFrame(watermark_candidates)
n_watermark = len(df_watermark)

print(f"\n {n_watermark} gambar terindikasi memiliki watermark/teks "
      f"di sudut ({100*n_watermark/len(df_stats_all):.2f}% dari total dataset).")

if n_watermark > 0:
    print(df_watermark["source"].value_counts().to_string())

    n_show = min(10, n_watermark)
    sample_wm = df_watermark.sample(n_show, random_state=SEED) if n_watermark > n_show else df_watermark

    fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2.2, 2.8))
    if n_show == 1:
        axes = [axes]
    fig.suptitle(f"Kandidat Gambar Ber-Watermark ({n_show} dari {n_watermark})",
                 fontweight='bold')

    for ax, (_, row) in zip(axes, sample_wm.iterrows()):
        data = np.fromfile(row["path"], dtype=np.uint8)
        img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(row["sudut_terdampak"], fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    _p = os.path.join(OUTPUT_DIR, 'audit_watermark_candidates.png')
    plt.savefig(_p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {_p}")

    _csv = os.path.join(OUTPUT_DIR, 'watermark_candidates.csv')
    df_watermark.to_csv(_csv, index=False)
    print(f"Daftar lengkap kandidat watermark: {_csv}")
else:
    print("Tidak ditemukan indikasi watermark pada dataset.")

In [ ]:
# ==============================================================================
# BAGIAN 3F — DETEKSI KANDIDAT CITRA CT SCAN (KONTAMINASI MODALITAS)
# ==============================================================================

CT_CONTRAST_PERCENTILE = 0.95   # ambang: 5% teratas contrast_std sbg kandidat awal
CT_BRIGHT_THRESH  = 230          # ambang "hampir putih murni"
CT_DARK_THRESH    = 15           # ambang "hampir hitam murni"
CT_BRIGHT_FRAC_MIN = 0.12        # minimal proporsi piksel sangat terang
CT_DARK_FRAC_MIN   = 0.35        # minimal proporsi piksel sangat gelap

def compute_bimodal_score(img_gray: np.ndarray):
    """Hitung proporsi piksel sangat terang & sangat gelap (indikasi bimodal CT)."""
    frac_bright = float(np.mean(img_gray > CT_BRIGHT_THRESH))
    frac_dark   = float(np.mean(img_gray < CT_DARK_THRESH))
    return frac_bright, frac_dark

print("=" * 65)
print("  DETEKSI KANDIDAT CITRA CT SCAN (KONTAMINASI MODALITAS)")
print("=" * 65)

# Ambil kandidat awal: 5% teratas berdasarkan contrast_std (dari BAGIAN 3D)
contrast_cutoff = df_stats_all["contrast_std"].quantile(CT_CONTRAST_PERCENTILE)
df_high_contrast = df_stats_all[df_stats_all["contrast_std"] >= contrast_cutoff].copy()

print(f"[INFO] Kandidat awal (contrast_std >= P{int(CT_CONTRAST_PERCENTILE*100)} "
      f"= {contrast_cutoff:.1f}): {len(df_high_contrast)} gambar")

ct_candidates = []
for _, row in tqdm(df_high_contrast.iterrows(), total=len(df_high_contrast),
                    desc="Cek pola bimodal CT"):
    data = np.fromfile(row["path"], dtype=np.uint8)
    img_gray = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        continue

    frac_bright, frac_dark = compute_bimodal_score(img_gray)

    if frac_bright >= CT_BRIGHT_FRAC_MIN and frac_dark >= CT_DARK_FRAC_MIN:
        ct_candidates.append({
            "path": row["path"],
            "source": row["source"],
            "contrast_std": row["contrast_std"],
            "frac_bright": round(frac_bright, 3),
            "frac_dark": round(frac_dark, 3),
        })

df_ct_candidates = pd.DataFrame(ct_candidates)
n_ct = len(df_ct_candidates)

print(f"\n{n_ct} gambar terindikasi sebagai kandidat CT scan "
      f"({100*n_ct/len(df_stats_all):.2f}% dari total dataset).")

if n_ct > 0:
    print("\nDistribusi sumber:")
    print(df_ct_candidates["source"].value_counts().to_string())

    n_show = min(10, n_ct)
    sample_ct = df_ct_candidates.sort_values("contrast_std", ascending=False).head(n_show)

    fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2.2, 2.8))
    if n_show == 1:
        axes = [axes]
    fig.suptitle(f"Kandidat Citra CT Scan (Top {n_show} dari {n_ct}, "
                 f"diurutkan berdasarkan contrast_std)", fontweight='bold')

    for ax, (_, row) in zip(axes, sample_ct.iterrows()):
        data = np.fromfile(row["path"], dtype=np.uint8)
        img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"c={row['contrast_std']:.0f}\nbr%={row['frac_bright']:.0%} "
                     f"dk%={row['frac_dark']:.0%}", fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    _p = os.path.join(OUTPUT_DIR, 'ct_scan_candidates.png')
    plt.savefig(_p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {_p}")

    _csv = os.path.join(OUTPUT_DIR, 'ct_scan_candidates.csv')
    df_ct_candidates.to_csv(_csv, index=False)
    print(f"Daftar lengkap kandidat CT scan: {_csv}")
else:
    print("Tidak ditemukan indikasi kontaminasi CT scan pada dataset.")

In [ ]:
# ==============================================================================
# BAGIAN 3G — PENGHAPUSAN GAMBAR BER-WATERMARK
# ==============================================================================
print("=" * 65)
print("  PENGHAPUSAN GAMBAR BER-WATERMARK (TERVERIFIKASI)")
print("=" * 65)

df_watermark["n_sudut"] = df_watermark["sudut_terdampak"].apply(
    lambda x: len(x.split(", "))
)

df_wm_high_conf = df_watermark[df_watermark["n_sudut"] >= 2].copy()
df_wm_low_conf  = df_watermark[df_watermark["n_sudut"] == 1].copy()

print(f"KLASIFIKASI CONFIDENCE")
print(f"  Confidence TINGGI (>=2 sudut) : {len(df_wm_high_conf)} gambar -> DIHAPUS")
print(f"  Confidence RENDAH (1 sudut)   : {len(df_wm_low_conf)} gambar -> DIBIARKAN")

# Hapus permanen file confidence tinggi dari direktori kerja (writable)
n_deleted, n_failed = 0, 0
for p in tqdm(df_wm_high_conf["path"], desc="Menghapus watermark confidence tinggi"):
    try:
        os.remove(p)
        n_deleted += 1
    except OSError:
        n_failed += 1

print(f"\nHASIL PENGHAPUSAN")
print(f"  Berhasil dihapus : {n_deleted}")
print(f"  Gagal dihapus    : {n_failed}")

# Visualisasi ringkasan sebelum vs sesudah penghapusan watermark
fig, ax = plt.subplots(figsize=(6, 4.5))
categories = ['Dihapus\n(confidence tinggi)', 'Dibiarkan\n(confidence rendah)']
counts = [n_deleted, len(df_wm_low_conf)]
colors = ['#e74c3c', '#95a5a6']
ax.bar(categories, counts, color=colors)
for i, c in enumerate(counts):
    ax.text(i, c + 1, str(c), ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Jumlah Gambar')
ax.set_title('Ringkasan Penghapusan Gambar Ber-Watermark', fontweight='bold')
plt.tight_layout()
_p = os.path.join(OUTPUT_DIR, 'watermark_removal_summary.png')
plt.savefig(_p, dpi=130, bbox_inches='tight')
plt.show()
print(f"[DISIMPAN] {_p}")

print(f"\nTotal dataset berkurang {n_deleted} gambar ber-watermark "
      f"(confidence tinggi) dari pipeline training.")

In [ ]:
# ==============================================================================
# BAGIAN 4 — EKSPLORASI STRUKTUR DIREKTORI DATASET
# ==============================================================================

def explore_dir(path, depth=2, prefix=""):
    if depth == 0:
        return
    try:
        entries = sorted(os.listdir(path))
    except NotADirectoryError:
        return
    for i, entry in enumerate(entries[:10]):
        full = os.path.join(path, entry)
        connector = "└── " if i == len(entries[:10]) - 1 else "├── "
        print(prefix + connector + entry)
        if os.path.isdir(full):
            explore_dir(full, depth - 1,
                        prefix + ("    " if i == len(entries[:10]) - 1 else "│   "))

print("=== STRUKTUR DATASET UTAMA ===")
explore_dir(PATH_MAIN, depth=3)

print("\n=== STRUKTUR DATASET EXTRA ===")
explore_dir(PATH_EXTRA, depth=3)

print("\n=== STRUKTUR DATASET BRATS (sample) ===")
explore_dir(PATH_BRATS, depth=3)

In [ ]:
# ==============================================================================
# BAGIAN 4B — VISUALISASI JUMLAH FILE PER KELAS PER DATASET (HASIL EKSPLORASI)
# ==============================================================================
# Melengkapi eksplorasi struktur direktori (teks) di atas dengan bentuk visual,
# menghitung jumlah file per subfolder kelas pada tiap dataset.

def count_per_class_folder(base_path, class_map=None):
    counts = {}
    for sf in ['Training', 'Testing', 'train', 'test', 'Train', 'Test']:
        p = os.path.join(base_path, sf)
        if not os.path.exists(p):
            continue
        for folder in sorted(os.listdir(p)):
            fp = os.path.join(p, folder)
            if not os.path.isdir(fp):
                continue
            n = len([f for f in os.listdir(fp)\
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            counts[folder] = counts.get(folder, 0) + n
    if not counts:
        for folder in sorted(os.listdir(base_path)):
            fp = os.path.join(base_path, folder)
            if not os.path.isdir(fp):
                continue
            n = len([f for f in os.listdir(fp)\
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            counts[folder] = counts.get(folder, 0) + n
    return counts

_counts_main_struct  = count_per_class_folder(PATH_MAIN)
_counts_extra_struct = count_per_class_folder(PATH_EXTRA)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, counts, title in zip(
    axes,
    [_counts_main_struct, _counts_extra_struct],
    ['Dataset Utama (MAIN)', 'Dataset Tambahan (EXTRA)']
):
    keys = list(counts.keys())
    vals = list(counts.values())
    ax.bar(keys, vals, color='#1abc9c')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Jumlah File')
    ax.tick_params(axis='x', rotation=45)
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals)*0.01, str(v), ha='center', fontsize=8)

fig.suptitle('Jumlah File per Folder Kelas Hasil Eksplorasi Struktur Direktori',
             fontsize=13, fontweight='bold')
plt.tight_layout()
_p_struct = os.path.join(OUTPUT_DIR, 'struktur_jumlah_file_per_kelas.png')
plt.savefig(_p_struct, dpi=130, bbox_inches='tight')
plt.show()
print(f"[DISIMPAN] {_p_struct}")

In [ ]:
# ==============================================================================
# BAGIAN 5 — FUNGSI PREPROCESSING UMUM
# ==============================================================================

def apply_clahe(image: np.ndarray) -> np.ndarray:
    """CLAHE di ruang warna LAB. Input/Output: RGB uint8."""
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_eq  = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2RGB)


def crop_brain_roi(image: np.ndarray) -> np.ndarray:
    """Contour Cropping: hapus area hitam di sekitar otak."""
    gray   = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray   = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thr = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    thr    = cv2.erode(thr, None, iterations=2)
    thr    = cv2.dilate(thr, None, iterations=2)
    cnts, _ = cv2.findContours(thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return image
    c = max(cnts, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)
    if w < 10 or h < 10:
        return image
    return image[y:y + h, x:x + w]


def preprocess_image(image: np.ndarray) -> np.ndarray:
    """
    Pipeline preprocessing standar:
      1. Gaussian blur ringan
      2. CLAHE
      3. Contour Cropping
      4. Resize → 128×128
      5. Normalisasi → [0, 1] float32
    """
    img = cv2.GaussianBlur(image, (3, 3), 0)
    img = apply_clahe(img)
    img = crop_brain_roi(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.astype('float32') / 255.0


print("Fungsi preprocessing siap.")

In [ ]:
# ==============================================================================
# BAGIAN 6 — FUNGSI PREPROCESSING BRATS (.nii.gz → 2D SLICE)
# ==============================================================================

def extract_brats_slice(nii_path: str) -> np.ndarray | None:
    """
    Ekstrak satu slice 2D terbaik dari volume MRI NIfTI.
    Return: float32 [0,1] shape (128,128,3) atau None.
    """
    try:
        vol = nib.load(nii_path).get_fdata()
        if vol.ndim == 4:
            vol = vol[..., 0]

        D       = vol.shape[2]
        z_start = int(D * 0.40)
        z_end   = int(D * 0.60)

        best_slice, best_score = None, -1
        for z in range(z_start, z_end):
            sl    = vol[:, :, z].astype('float32')
            score = np.count_nonzero(sl)
            if score > best_score:
                best_score = score
                best_slice = sl

        if best_slice is None or best_score == 0:
            return None

        mn, mx = best_slice.min(), best_slice.max()
        if mx - mn < 1e-8:
            return None
        sl_norm = ((best_slice - mn) / (mx - mn) * 255).astype('uint8')
        sl_rgb  = cv2.cvtColor(sl_norm, cv2.COLOR_GRAY2RGB)
        return preprocess_image(sl_rgb)

    except Exception:
        return None


def load_brats_dataset(brats_root: str, max_per_class: int = 300):
    """
    Load dataset BraTS untuk external validation.
    Semua sampel BraTS = kelas 0 (glioma).
    Dibatasi max_per_class agar tidak OOM.
    """
    images, labels = [], []

    print("\n" + "=" * 65)
    print("  MEMUAT DATASET BRATS (.nii.gz → 2D slice)")
    print("=" * 65)

    training_dir = os.path.join(
        brats_root, 'BraTS2020_TrainingData', 'MICCAI_BraTS2020_TrainingData'
    )
    if not os.path.exists(training_dir):
        training_dir = brats_root
        print(f"Mencari file .nii.gz di: {training_dir}")

    t1ce_files = glob.glob(
        os.path.join(training_dir, '**', '*t1ce*'), recursive=True
    )
    t1ce_files = [f for f in t1ce_files\
                  if f.endswith('.nii') or f.endswith('.nii.gz')]

    if not t1ce_files:
        t1ce_files = glob.glob(
            os.path.join(training_dir, '**', '*.nii.gz'), recursive=True
        )
        t1ce_files = t1ce_files[:max_per_class]
        print(f"T1ce tidak ditemukan, menggunakan {len(t1ce_files)} file .nii.gz")
    else:
        print(f"Ditemukan {len(t1ce_files)} file T1ce")

    loaded = 0
    for fpath in tqdm(t1ce_files, desc="Proses BraTS"):
        if loaded >= max_per_class:
            break
        img = extract_brats_slice(fpath)
        if img is not None:
            images.append(img)
            labels.append(0)
            loaded += 1

    print(f"BraTS: {loaded} slice berhasil dimuat (kelas: glioma)")

    if not images:
        return (np.empty((0, IMG_SIZE, IMG_SIZE, CHANNELS), dtype='float32'),
                np.empty(0, dtype='int32'))

    return np.array(images, dtype='float32'), np.array(labels, dtype='int32')


print("Fungsi preprocessing BraTS siap.")

In [ ]:
# ==============================================================================
# BAGIAN 7 — KUMPULKAN FILE PATH (tf.data, TANPA LOAD KE RAM)
# ==============================================================================

CLASS_MAP = {
    'glioma': 0, 'glioma_tumor': 0,
    'meningioma': 1, 'meningioma_tumor': 1,
    'no_tumor': 2, 'notumor': 2, 'no tumor': 2, 'normal': 2,
    'pituitary': 3, 'pituitary_tumor': 3,
}


def collect_file_paths(base_path: str, source_name: str = "Dataset"):
    records = []
    class_counts = {c: 0 for c in CLASSES}

    subfolders = []
    for sf in ['Training', 'Testing', 'train', 'test', 'Train', 'Test']:
        p = os.path.join(base_path, sf)
        if os.path.exists(p):
            subfolders.append(p)
    if not subfolders:
        subfolders = [base_path]

    print(f"\n[{source_name}] Mengumpulkan path dari: {base_path}")

    for sf in subfolders:
        for folder in sorted(os.listdir(sf)):
            folder_lower = folder.lower()
            label_idx    = CLASS_MAP.get(folder_lower)
            if label_idx is None:
                continue
            folder_path = os.path.join(sf, folder)
            if not os.path.isdir(folder_path):
                continue
            files = sorted([f for f in os.listdir(folder_path)\
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            for fname in files:
                records.append((os.path.join(folder_path, fname), label_idx))
                class_counts[CLASSES[label_idx]] += 1

    print(f"  Total: {len(records)} gambar")
    for cls, cnt in class_counts.items():
        print(f"    {cls:12s}: {cnt}")

    return records


print("\n" + "=" * 65)
print("  MENGUMPULKAN PATH DATASET UTAMA")
print("=" * 65)
records_main = collect_file_paths(PATH_MAIN, "MAIN")

print("\n" + "=" * 65)
print("  MENGUMPULKAN PATH DATASET TAMBAHAN")
print("=" * 65)
records_extra = collect_file_paths(PATH_EXTRA, "EXTRA")

In [ ]:
# ==============================================================================
# BAGIAN 8 — GABUNGKAN & SPLIT PATH
# ==============================================================================

all_records = records_main + records_extra
random.seed(SEED)
random.shuffle(all_records)

all_paths  = [r[0] for r in all_records]
all_labels = [r[1] for r in all_records]

print(f"\n Total: {len(all_paths)} gambar dari dataset utama + extra")
for i, cls in enumerate(CLASSES):
    cnt = sum(1 for l in all_labels if l == i)
    print(f"  {cls:12s}: {cnt}")

# Split 70/15/15
paths_train, paths_temp, y_train_raw, y_temp = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=SEED,
    stratify=all_labels
)
paths_val, paths_test, y_val_raw, y_test_raw = train_test_split(
    paths_temp, y_temp, test_size=0.50, random_state=SEED,
    stratify=y_temp
)

y_train = np.array(y_train_raw, dtype='int32')
y_val   = np.array(y_val_raw,   dtype='int32')
y_test  = np.array(y_test_raw,  dtype='int32')

print(f"\nSPLIT Train: {len(paths_train)} | Val: {len(paths_val)} | Test: {len(paths_test)}")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:12s}: Train={( y_train==i).sum():5d} | "
          f"Val={(y_val==i).sum():4d} | Test={(y_test==i).sum():4d}")

In [ ]:
# ==============================================================================
# BAGIAN 8B — VISUALISASI 10 DATA PERTAMA & 10 DATA TERAKHIR (RAW / SEBELUM
#              PREPROCESSING)
# ==============================================================================
def show_raw_first_last(records, classes, n=10, title_prefix="RAW"):
    first_n = records[:n]
    last_n  = records[-n:]

    fig, axes = plt.subplots(2, n, figsize=(n * 2.2, 5))
    fig.suptitle(f"Visualisasi Data {title_prefix}: {n} Data Pertama & {n} Data Terakhir",
                 fontsize=13, fontweight='bold')

    for row_idx, (group, label) in enumerate([(first_n, f"{n} Data Pertama"),\
                                                (last_n, f"{n} Data Terakhir")]):
        for col_idx, (path, cls_idx) in enumerate(group):
            data = np.fromfile(path, dtype=np.uint8)
            img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
            img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            ax = axes[row_idx, col_idx]
            ax.imshow(img)
            ax.set_title(f"{classes[cls_idx]}\n{img.shape[1]}x{img.shape[0]}", fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])

            if col_idx == 0:
                ax.set_ylabel(label, fontsize=10, fontweight='bold')

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, 'raw_first10_last10.png')
    plt.savefig(p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p}")

print("=" * 65)
print("  VISUALISASI DATA RAW: 10 PERTAMA & 10 TERAKHIR")
print("=" * 65)
print(f"Total data pada all_records (setelah cleaning, sebelum split): {len(all_records)}")

show_raw_first_last(all_records, CLASSES, n=10)

In [ ]:
# ==============================================================================
# BAGIAN 8C — 10 DATA PERTAMA & 10 DATA TERAKHIR PER KELAS TUMOR (RAW)
# ==============================================================================
# Berbeda dengan Bagian 8B (gabungan seluruh kelas secara acak/shuffle), sel ini
# menampilkan 10 data pertama & 10 data terakhir UNTUK SETIAP KELAS tumor otak
# secara terpisah (glioma, meningioma, no_tumor, pituitary), sesuai urutan file
# hasil collect_file_paths (sebelum shuffle/split).

def show_raw_first_last_per_class(records, classes, n=10):
    for cls_idx, cls_name in enumerate(classes):
        cls_records = [r for r in records if r[1] == cls_idx]
        if len(cls_records) < 1:
            print(f"[LEWATI] Kelas {cls_name}: tidak ada data.")
            continue

        first_n = cls_records[:n]
        last_n  = cls_records[-n:]

        fig, axes = plt.subplots(2, n, figsize=(n * 2.0, 4.6))
        fig.suptitle(f"Kelas: {cls_name.upper()} — {n} Data Pertama & {n} Data Terakhir (Raw)",
                     fontsize=13, fontweight='bold')

        for row_idx, (group, label) in enumerate([(first_n, f"{n} Pertama"),\
                                                    (last_n, f"{n} Terakhir")]):
            for col_idx in range(n):
                ax = axes[row_idx, col_idx]
                if col_idx < len(group):
                    path, _ = group[col_idx]
                    data = np.fromfile(path, dtype=np.uint8)
                    img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
                    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    ax.imshow(img)
                    ax.set_title(f"{img.shape[1]}x{img.shape[0]}", fontsize=8)
                ax.set_xticks([]); ax.set_yticks([])
                if col_idx == 0:
                    ax.set_ylabel(label, fontsize=10, fontweight='bold')

        plt.tight_layout()
        p = os.path.join(OUTPUT_DIR, f'raw_first10_last10_{cls_name}.png')
        plt.savefig(p, dpi=130, bbox_inches='tight')
        plt.show()
        print(f"[DISIMPAN] {p}")

print("=" * 65)
print("  VISUALISASI DATA RAW PER KELAS: 10 PERTAMA & 10 TERAKHIR")
print("=" * 65)
show_raw_first_last_per_class(all_records, CLASSES, n=10)

In [ ]:
# ==============================================================================
# BAGIAN 9 — LOAD DATASET BRATS
# ==============================================================================

X_brats, y_brats = load_brats_dataset(PATH_BRATS, max_per_class=300)

if len(X_brats) > 0:
    print(f"\n{len(X_brats)} slice siap untuk external validation")
    print(f"Semua label = glioma (kelas 0)")
else:
    print("BraTS tidak berhasil dimuat. External validation dilewati.")

In [ ]:
# ==============================================================================
# BAGIAN 9B — VISUALISASI VOLUME 3D BRATS (.NII.GZ) vs SLICE 2D HASIL EKSTRAKSI
# ==============================================================================
# Menunjukkan data BraTS 2020 dalam bentuk aslinya (volume MRI 3D, format
# .nii.gz) sebelum diproses, kemudian dibandingkan dengan satu slice 2D
# terbaik hasil fungsi extract_brats_slice() yang dipakai dalam pipeline.

def find_sample_brats_file(brats_root):
    training_dir = os.path.join(
        brats_root, 'BraTS2020_TrainingData', 'MICCAI_BraTS2020_TrainingData'
    )
    if not os.path.exists(training_dir):
        training_dir = brats_root
    t1ce_files = glob.glob(os.path.join(training_dir, '**', '*t1ce*'), recursive=True)
    t1ce_files = [f for f in t1ce_files if f.endswith('.nii') or f.endswith('.nii.gz')]
    if not t1ce_files:
        t1ce_files = glob.glob(os.path.join(training_dir, '**', '*.nii.gz'), recursive=True)
    return t1ce_files[0] if t1ce_files else None

def visualize_brats_volume_vs_slice(nii_path, n_montage=9):
    vol = nib.load(nii_path).get_fdata()
    if vol.ndim == 4:
        vol = vol[..., 0]
    D = vol.shape[2]

    # (A) Montase axial slice dari volume 3D lengkap (menyebar dari depan ke belakang)
    idxs = np.linspace(0, D - 1, n_montage).astype(int)
    cols = n_montage
    fig, axes = plt.subplots(1, cols, figsize=(cols * 2.0, 2.6))
    fig.suptitle(f"Volume MRI 3D BraTS (.nii.gz) — Montase {n_montage} Axial Slice "
                 f"dari Total {D} Slice\n{os.path.basename(nii_path)}",
                 fontsize=12, fontweight='bold')
    for i, z in enumerate(idxs):
        axes[i].imshow(vol[:, :, z], cmap='gray')
        axes[i].set_title(f"z={z}", fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    p1 = os.path.join(OUTPUT_DIR, 'brats_3d_volume_montage.png')
    plt.savefig(p1, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p1}")

    # (B) 3 bidang ortogonal (axial, coronal, sagittal) dari volume 3D
    cx, cy, cz = vol.shape[0] // 2, vol.shape[1] // 2, vol.shape[2] // 2
    fig, axes = plt.subplots(1, 3, figsize=(10, 4))
    axes[0].imshow(vol[:, :, cz], cmap='gray'); axes[0].set_title(f"Axial (z={cz})")
    axes[1].imshow(vol[:, cy, :], cmap='gray'); axes[1].set_title(f"Coronal (y={cy})")
    axes[2].imshow(vol[cx, :, :], cmap='gray'); axes[2].set_title(f"Sagittal (x={cx})")
    for ax in axes:
        ax.axis('off')
    fig.suptitle("Tiga Bidang Ortogonal Volume MRI 3D BraTS", fontsize=12, fontweight='bold')
    plt.tight_layout()
    p2 = os.path.join(OUTPUT_DIR, 'brats_3d_orthogonal_planes.png')
    plt.savefig(p2, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p2}")

    # (C) Perbandingan langsung: slice mentah 3D terbaik vs hasil extract_brats_slice (2D final)
    slice_2d_final = extract_brats_slice(nii_path)
    z_start, z_end = int(D * 0.40), int(D * 0.60)
    best_slice, best_score = None, -1
    for z in range(z_start, z_end):
        sl = vol[:, :, z]
        score = np.count_nonzero(sl)
        if score > best_score:
            best_score, best_slice = score, sl

    fig, axes = plt.subplots(1, 2, figsize=(8, 4.3))
    axes[0].imshow(best_slice, cmap='gray')
    axes[0].set_title(f"Slice 3D Mentah Terpilih (z={z_start}-{z_end})", fontsize=10)
    axes[0].axis('off')
    if slice_2d_final is not None:
        axes[1].imshow(slice_2d_final)
        axes[1].set_title("Setelah extract_brats_slice()\n+ preprocess_image()\n(128x128, RGB, [0,1])",
                           fontsize=10)
    axes[1].axis('off')
    fig.suptitle("Perbandingan: Slice Mentah dari Volume 3D vs Slice 2D Final Hasil Ekstraksi",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    p3 = os.path.join(OUTPUT_DIR, 'brats_3d_vs_2d_final.png')
    plt.savefig(p3, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p3}")

_sample_nii = find_sample_brats_file(PATH_BRATS)
if _sample_nii:
    print(f"Sampel volume BraTS untuk visualisasi: {_sample_nii}")
    visualize_brats_volume_vs_slice(_sample_nii)
else:
    print("[PERINGATAN] Tidak ditemukan file .nii.gz sampel untuk visualisasi.")

In [ ]:
# ==============================================================================
# BAGIAN 10 — tf.data PIPELINE & MIXED TRAINING (DOMAIN INJECTION)
# ==============================================================================

def load_and_preprocess_tf(path: tf.Tensor, label: tf.Tensor):
    """Baca satu file gambar, preprocess, return (img_float32, label)."""
    def _load(p):
        img = cv2.imdecode(
            np.frombuffer(open(p.numpy().decode(), 'rb').read(), np.uint8),
            cv2.IMREAD_COLOR
        )
        if img is None:
            return np.zeros((IMG_SIZE, IMG_SIZE, CHANNELS), dtype=np.float32)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return preprocess_image(img).astype(np.float32)

    img = tf.py_function(_load, [path], tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, CHANNELS])
    return img, label

# Augmentasi untuk classifier
aug_clf = tf.keras.Sequential([\
    tf.keras.layers.RandomFlip('horizontal'),\
    tf.keras.layers.RandomRotation(0.08, fill_mode='constant', fill_value=0.0),\
    tf.keras.layers.RandomZoom(0.08, fill_mode='constant', fill_value=0.0),\
    tf.keras.layers.RandomTranslation(0.07, 0.07, fill_mode='constant', fill_value=0.0),\
    tf.keras.layers.RandomBrightness(0.10, value_range=(0.0, 1.0)),\
    tf.keras.layers.RandomContrast(0.10),\
], name='aug_clf')

# Augmentasi ringan untuk autoencoder
aug_ae = tf.keras.Sequential([\
    tf.keras.layers.RandomFlip('horizontal'),\
    tf.keras.layers.RandomRotation(0.05, fill_mode='constant', fill_value=0.0),\
], name='aug_ae')

def make_clf_dataset(paths, labels, training=True):
    """Dataset pipeline standar untuk validation/testing murni."""
    ds = tf.data.Dataset.from_tensor_slices((tf.constant(paths), tf.constant(labels, dtype=tf.int32)))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(load_and_preprocess_tf, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(
            lambda x, lbl: (aug_clf(x[tf.newaxis], training=True)[0], lbl),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def make_ae_dataset(paths, training=True):
    """Dataset pipeline standar untuk validation Autoencoder murni."""
    dummy_labels = tf.zeros(len(paths), dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((tf.constant(paths), dummy_labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(load_and_preprocess_tf, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        def augment_pair(img, _):
            aug_img = aug_ae(img[tf.newaxis], training=True)[0]
            return aug_img, aug_img
        ds = ds.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda img, _: (img, img), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# 1. Ambil 20% BraTS untuk training
porsi_injeksi = 0.20
num_brats_train = int(len(X_brats) * porsi_injeksi)

X_brats_train = X_brats[:num_brats_train].astype(np.float32)
X_brats_test  = X_brats[num_brats_train:].astype(np.float32)

# Label Glioma = 0
y_brats_train = np.zeros(len(X_brats_train), dtype=np.int32)
y_brats_test  = np.zeros(len(X_brats_test), dtype=np.int32)

# 2. Bikin Dataset Classifier Gabungan (Kaggle Path + BraTS Array)
ds_kaggle_clf = tf.data.Dataset.from_tensor_slices((tf.constant(paths_train), tf.constant(y_train_raw, dtype=tf.int32)))
ds_kaggle_clf = ds_kaggle_clf.map(load_and_preprocess_tf, num_parallel_calls=tf.data.AUTOTUNE)

ds_brats_clf = tf.data.Dataset.from_tensor_slices((X_brats_train, y_brats_train))

combined_clf_ds = ds_kaggle_clf.concatenate(ds_brats_clf).shuffle(2048, seed=SEED)
clf_train_ds = combined_clf_ds.map(
    lambda x, lbl: (aug_clf(x[tf.newaxis], training=True)[0], lbl),
    num_parallel_calls=tf.data.AUTOTUNE
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# 3. Bikin Dataset Autoencoder Gabungan (Kaggle Path + BraTS Array)
paths_ae = paths_train
n_ae_val = int(len(paths_ae) * 0.10)
paths_ae_train = paths_ae[n_ae_val:]
paths_ae_val   = paths_ae[:n_ae_val]

ds_kaggle_ae = tf.data.Dataset.from_tensor_slices((tf.constant(paths_ae_train), tf.zeros(len(paths_ae_train), dtype=tf.int32)))
ds_kaggle_ae = ds_kaggle_ae.map(load_and_preprocess_tf, num_parallel_calls=tf.data.AUTOTUNE)

ds_brats_ae = tf.data.Dataset.from_tensor_slices((X_brats_train, tf.zeros(len(X_brats_train), dtype=tf.int32)))

def augment_pair_ae(img, _):
    aug_img = aug_ae(img[tf.newaxis], training=True)[0]
    return aug_img, aug_img

combined_ae_ds = ds_kaggle_ae.concatenate(ds_brats_ae).shuffle(2048, seed=SEED)
ae_train_ds = combined_ae_ds.map(augment_pair_ae, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# 4. Dataset Validation & Test (Murni)
clf_val_ds   = make_clf_dataset(paths_val,   y_val_raw,   training=False)
clf_test_ds  = make_clf_dataset(paths_test,  y_test_raw,  training=False)
ae_val_ds    = make_ae_dataset(paths_ae_val,   training=False)

print("Pipeline selesai. Mixed Training (Domain Injection) aktif.")
print(f"  -> Total Latih Classifier : {len(paths_train) + len(X_brats_train)} (Kaggle + BraTS)")
print(f"  -> Sisa Uji BraTS         : {len(X_brats_test)}")

In [ ]:
# ==============================================================================
# BAGIAN 11 — VISUALISASI SAMPEL DATA
# ==============================================================================

def visualize_samples_from_ds(ds, title="Sample", n=12):
    imgs, lbls = [], []
    for batch_imgs, batch_lbls in ds.take(2):
        imgs.append(batch_imgs.numpy())
        lbls.append(batch_lbls.numpy())
    imgs = np.concatenate(imgs)[:n]
    lbls = np.concatenate(lbls)[:n]

    cols = min(n, 6)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    axes = np.array(axes).flatten()

    for i in range(n):
        axes[i].imshow(np.clip(imgs[i], 0.0, 1.0))
        axes[i].set_title(CLASSES[lbls[i]], fontsize=9)
        axes[i].axis('off')

    for i in range(n, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'samples_{title.replace(" ", "_")}.png'),
                dpi=100, bbox_inches='tight')
    plt.show()
visualize_samples_from_ds(clf_train_ds, "Sampel Data Training")

In [ ]:
# ==============================================================================
# BAGIAN 11B — VISUALISASI SAMPEL PER KELAS
# ==============================================================================

def show_class_samples(records, classes, n_per_class=4, title="Sampel Data per Kelas"):
    """
    records      : list of (path, label_int)
    classes      : daftar nama kelas sesuai indeks label
    n_per_class  : jumlah sampel gambar per kelas yang ditampilkan
    """
    fig, axes = plt.subplots(len(classes), n_per_class,
                              figsize=(n_per_class * 3, len(classes) * 3))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    for i, cls_name in enumerate(classes):
        cls_paths = [p for p, l in records if l == i]
        chosen    = random.sample(cls_paths, min(n_per_class, len(cls_paths)))

        for j in range(n_per_class):
            ax = axes[i, j]
            if j < len(chosen):
                data = np.fromfile(chosen[j], dtype=np.uint8)
                img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
                img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img)
            if j == 0:
                ax.set_ylabel(cls_name.capitalize(), fontsize=12, fontweight='bold')
            ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, 'eda_sampel_per_kelas.png')
    plt.savefig(p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p}")

show_class_samples(all_records, CLASSES, n_per_class=4,
                    title="Sampel Citra MRI per Kelas (Sebelum Preprocessing)")

In [ ]:
# ==============================================================================
# BAGIAN 11C — BUKTI PREPROCESSING: ORIGINAL vs CROP vs CLAHE (+ HISTOGRAM)
# ==============================================================================

def compare_preprocessing_stages(image_path: str, save_name: str = "eda_clahe_comparison.png"):
    data = np.fromfile(image_path, dtype=np.uint8)
    img_bgr = cv2.imdecode(data, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Tahap 1: Citra asli
    original = img_rgb

    # Tahap 2: Gaussian blur + contour cropping (sebelum CLAHE)
    blurred  = cv2.GaussianBlur(original, (3, 3), 0)
    cropped  = crop_brain_roi(blurred)

    # Tahap 3: CLAHE diterapkan pada hasil crop
    clahe_img = apply_clahe(cropped)

    stages = [\
        ("1. Citra Asli (Original)", original),\
        ("2. Hasil Contour Cropping", cropped),\
        ("3. Hasil CLAHE", clahe_img),\
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("Bukti Efek Preprocessing: Original vs Crop vs CLAHE",
                  fontsize=14, fontweight='bold')

    for col, (label, im) in enumerate(stages):
        # Baris atas: gambar
        axes[0, col].imshow(im)
        axes[0, col].set_title(label, fontsize=11)
        axes[0, col].axis('off')

        # Baris bawah: histogram grayscale
        gray = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)
        axes[1, col].hist(gray.ravel(), bins=256, range=(0, 256),
                           color='steelblue', alpha=0.8)
        axes[1, col].set_title(f"Histogram - {label}", fontsize=10)
        axes[1, col].set_xlabel("Intensitas Piksel")
        axes[1, col].set_ylabel("Frekuensi")
        axes[1, col].set_xlim(0, 255)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, save_name)
    plt.savefig(p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p}")

# Ambil satu sampel acak dari dataset untuk demonstrasi
_sample_path = random.choice(all_paths)
compare_preprocessing_stages(_sample_path)

In [ ]:
# ==============================================================================
# BAGIAN 11C-2 — VISUALISASI LENGKAP SETIAP TAHAP PREPROCESSING (SESUAI URUTAN
#                 ASLI PIPELINE preprocess_image())
# ==============================================================================
# Melengkapi Bagian 11C (yang hanya menunjukkan Original vs Crop vs CLAHE)
# dengan SELURUH tahap preprocessing satu per satu, PERSIS sesuai urutan yang
# dijalankan oleh preprocess_image():
#   1. Citra Asli (Original)
#   2. Gaussian Blur (3x3)
#   3. CLAHE (LAB color space)
#   4. Contour Cropping (hapus area hitam di sekitar otak)
#   5. Resize -> 128x128
#   6. Normalisasi -> [0, 1] float32

def visualize_full_preprocessing_pipeline(image_path, save_name='pipeline_lengkap_semua_tahap.png'):
    data = np.fromfile(image_path, dtype=np.uint8)
    img_bgr = cv2.imdecode(data, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Tahap demi tahap, urutan PERSIS sama dengan preprocess_image()
    stage1_original = img_rgb
    stage2_blur     = cv2.GaussianBlur(stage1_original, (3, 3), 0)
    stage3_clahe    = apply_clahe(stage2_blur)
    stage4_crop     = crop_brain_roi(stage3_clahe)
    stage5_resize   = cv2.resize(stage4_crop, (IMG_SIZE, IMG_SIZE))
    stage6_norm     = stage5_resize.astype('float32') / 255.0

    stages = [\
        ("1. Citra Asli\n(Original)", stage1_original, f"{stage1_original.shape}\nuint8 [0,255]"),\
        ("2. Gaussian Blur\n(kernel 3x3)", stage2_blur, f"{stage2_blur.shape}\nuint8 [0,255]"),\
        ("3. CLAHE\n(ruang warna LAB)", stage3_clahe, f"{stage3_clahe.shape}\nuint8 [0,255]"),\
        ("4. Contour Cropping\n(hapus latar hitam)", stage4_crop, f"{stage4_crop.shape}\nuint8 [0,255]"),\
        ("5. Resize\n(128x128)", stage5_resize, f"{stage5_resize.shape}\nuint8 [0,255]"),\
        ("6. Normalisasi\n([0,1] float32)", stage6_norm, f"{stage6_norm.shape}\nfloat32 [0,1]"),\
    ]

    fig, axes = plt.subplots(2, len(stages), figsize=(len(stages) * 2.6, 6.5))
    fig.suptitle("Visualisasi Lengkap Seluruh Tahap Preprocessing (Sesuai Urutan Pipeline Asli)",
                 fontsize=14, fontweight='bold')

    for col, (label, im, info) in enumerate(stages):
        axes[0, col].imshow(np.clip(im, 0, 1) if im.dtype != np.uint8 else im)
        axes[0, col].set_title(label, fontsize=10)
        axes[0, col].set_xlabel(info, fontsize=8)
        axes[0, col].set_xticks([]); axes[0, col].set_yticks([])

        gray = cv2.cvtColor((im * 255).astype('uint8') if im.dtype != np.uint8 else im,
                             cv2.COLOR_RGB2GRAY)
        axes[1, col].hist(gray.ravel(), bins=256, range=(0, 256), color='darkorange', alpha=0.8)
        axes[1, col].set_title(f"Histogram - Tahap {col+1}", fontsize=9)
        axes[1, col].set_xlabel("Intensitas Piksel", fontsize=8)
        axes[1, col].set_xlim(0, 255)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, save_name)
    plt.savefig(p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p}")

    print("\nRingkasan transformasi shape/dtype per tahap:")
    for label, im, info in stages:
        print(f"  {label.splitlines()[0]:28s}: {info.splitlines()[0]:16s} {info.splitlines()[1]}")

print("=" * 65)
print("  VISUALISASI LENGKAP SEMUA TAHAP PREPROCESSING")
print("=" * 65)

# Tampilkan untuk 1 sampel per kelas agar representatif
_local_rng = random.Random(SEED)  # RNG lokal, TIDAK menyentuh state random global
for _cls_idx, _cls_name in enumerate(CLASSES):
    _cls_paths = [p for p, l in all_records if l == _cls_idx]
    if _cls_paths:
        _sample = _local_rng.choice(_cls_paths)
        print(f"\n--- Kelas: {_cls_name} ---")
        visualize_full_preprocessing_pipeline(
            _sample, save_name=f'pipeline_lengkap_semua_tahap_{_cls_name}.png'
        )

In [ ]:
# ==============================================================================
# BAGIAN 11D — AUDIT HASIL CONTOUR CROPPING PER KELAS
# ==============================================================================

N_AUDIT_SAMPLES_PER_CLASS = 5
N_STAT_SAMPLES_PER_CLASS  = 100

def audit_crop_visual(records, classes, n_per_class=N_AUDIT_SAMPLES_PER_CLASS):
    """Tampilkan grid: baris=kelas, kolom berpasangan (Original | Crop)."""
    fig, axes = plt.subplots(len(classes), n_per_class * 2,
                              figsize=(n_per_class * 4, len(classes) * 2.5))
    fig.suptitle("Audit Contour Cropping per Kelas (Original vs Hasil Crop)",
                  fontsize=14, fontweight='bold')

    for i, cls_name in enumerate(classes):
        cls_paths = [p for p, l in records if l == i]
        chosen    = random.sample(cls_paths, min(n_per_class, len(cls_paths)))

        for j, p in enumerate(chosen):
            data = np.fromfile(p, dtype=np.uint8)
            img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
            img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            blurred = cv2.GaussianBlur(img, (3, 3), 0)
            cropped = crop_brain_roi(blurred)

            ax_orig = axes[i, j * 2]
            ax_crop = axes[i, j * 2 + 1]

            ax_orig.imshow(img)
            ax_crop.imshow(cropped)

            for ax in (ax_orig, ax_crop):
                ax.set_xticks([]); ax.set_yticks([])

            if j == 0:
                ax_orig.set_ylabel(cls_name.capitalize(), fontsize=11, fontweight='bold')
            if i == 0:
                ax_orig.set_title("Original", fontsize=9)
                ax_crop.set_title("Crop", fontsize=9)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, 'audit_crop_per_kelas.png')
    plt.savefig(p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p}")

def audit_crop_statistics(records, classes, n_per_class=N_STAT_SAMPLES_PER_CLASS):
    """
    Hitung rasio (luas hasil crop / luas gambar asli) untuk subset acak per
    kelas. Rasio = 1.0 berarti crop gagal validasi dan fallback ke original.
    """
    random.seed(SEED)
    rows = []

    for i, cls_name in enumerate(classes):
        cls_paths = [p for p, l in records if l == i]
        chosen    = random.sample(cls_paths, min(n_per_class, len(cls_paths)))

        ratios = []
        n_fallback = 0
        for p in chosen:
            data = np.fromfile(p, dtype=np.uint8)
            img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
            img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            blurred = cv2.GaussianBlur(img, (3, 3), 0)
            cropped = crop_brain_roi(blurred)

            area_img  = img.shape[0] * img.shape[1]
            area_crop = cropped.shape[0] * cropped.shape[1]
            ratio = area_crop / area_img
            ratios.append(ratio)

            if ratio >= 0.999:
                n_fallback += 1

        rows.append({
            "Kelas": cls_name,
            "Jumlah Sampel": len(chosen),
            "Rasio Crop Rata-rata": round(float(np.mean(ratios)), 3),
            "Rasio Crop Min": round(float(np.min(ratios)), 3),
            "Rasio Crop Max": round(float(np.max(ratios)), 3),
            "Fallback ke Original (%)": round(100 * n_fallback / len(chosen), 1),
        })

    df = pd.DataFrame(rows)
    print("=" * 65)
    print("  STATISTIK AUDIT CONTOUR CROPPING")
    print("=" * 65)
    print(df.to_string(index=False))

    csv_path = os.path.join(OUTPUT_DIR, 'audit_crop_statistics.csv')
    df.to_csv(csv_path, index=False)
    print(f"\n[DISIMPAN] {csv_path}")
    return df

audit_crop_visual(all_records, CLASSES)
_ = audit_crop_statistics(all_records, CLASSES)

In [ ]:
# ==============================================================================
# BAGIAN 11E — VISUALISASI 10 DATA PERTAMA & 10 DATA TERAKHIR (SETELAH
#              PREPROCESSING)
# ==============================================================================
def show_processed_first_last(records, classes, n=10, title_prefix="Setelah Preprocessing"):
    first_n = records[:n]
    last_n  = records[-n:]

    fig, axes = plt.subplots(2, n, figsize=(n * 2.2, 5))
    fig.suptitle(f"Visualisasi Data {title_prefix}: {n} Data Pertama & {n} Data Terakhir",
                 fontsize=13, fontweight='bold')

    for row_idx, (group, label) in enumerate([(first_n, f"{n} Data Pertama"),\
                                                (last_n, f"{n} Data Terakhir")]):
        for col_idx, (path, cls_idx) in enumerate(group):
            data = np.fromfile(path, dtype=np.uint8)
            img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
            img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_processed = preprocess_image(img)  # hasil pipeline penuh, float32 [0,1]

            ax = axes[row_idx, col_idx]
            ax.imshow(img_processed)
            ax.set_title(f"{classes[cls_idx]}\n{IMG_SIZE}x{IMG_SIZE}", fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])

            if col_idx == 0:
                ax.set_ylabel(label, fontsize=10, fontweight='bold')

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, 'processed_first10_last10.png')
    plt.savefig(p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"[DISIMPAN] {p}")

print("=" * 65)
print("  VISUALISASI DATA SETELAH PREPROCESSING: 10 PERTAMA & 10 TERAKHIR")
print("=" * 65)

show_processed_first_last(all_records, CLASSES, n=10)

In [ ]:
# ==============================================================================
# BAGIAN 11F — 10 DATA PERTAMA & 10 DATA TERAKHIR PER KELAS (SETELAH PREPROCESSING)
# ==============================================================================
# Versi per-kelas dari Bagian 11E, menunjukkan hasil pipeline preprocess_image()
# penuh (Gaussian Blur -> CLAHE -> Contour Cropping -> Resize 128x128 -> Normalisasi)
# untuk 10 data pertama & 10 data terakhir pada MASING-MASING kelas tumor.

def show_processed_first_last_per_class(records, classes, n=10):
    for cls_idx, cls_name in enumerate(classes):
        cls_records = [r for r in records if r[1] == cls_idx]
        if len(cls_records) < 1:
            print(f"[LEWATI] Kelas {cls_name}: tidak ada data.")
            continue

        first_n = cls_records[:n]
        last_n  = cls_records[-n:]

        fig, axes = plt.subplots(2, n, figsize=(n * 2.0, 4.6))
        fig.suptitle(f"Kelas: {cls_name.upper()} — {n} Data Pertama & {n} Data Terakhir "
                     f"(Setelah Preprocessing)", fontsize=13, fontweight='bold')

        for row_idx, (group, label) in enumerate([(first_n, f"{n} Pertama"),\
                                                    (last_n, f"{n} Terakhir")]):
            for col_idx in range(n):
                ax = axes[row_idx, col_idx]
                if col_idx < len(group):
                    path, _ = group[col_idx]
                    data = np.fromfile(path, dtype=np.uint8)
                    img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
                    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img_processed = preprocess_image(img)
                    ax.imshow(img_processed)
                    ax.set_title(f"{IMG_SIZE}x{IMG_SIZE}", fontsize=8)
                ax.set_xticks([]); ax.set_yticks([])
                if col_idx == 0:
                    ax.set_ylabel(label, fontsize=10, fontweight='bold')

        plt.tight_layout()
        p = os.path.join(OUTPUT_DIR, f'processed_first10_last10_{cls_name}.png')
        plt.savefig(p, dpi=130, bbox_inches='tight')
        plt.show()
        print(f"[DISIMPAN] {p}")

print("=" * 65)
print("  VISUALISASI DATA SETELAH PREPROCESSING PER KELAS: 10 PERTAMA & 10 TERAKHIR")
print("=" * 65)
show_processed_first_last_per_class(all_records, CLASSES, n=10)

In [ ]:
# ==============================================================================
# BAGIAN 12 — CLASS WEIGHTS
# ==============================================================================

cw_array = compute_class_weight('balanced',
                                 classes=np.unique(y_train),
                                 y=y_train)
CLASS_WEIGHTS = dict(enumerate(cw_array))

print("[CLASS WEIGHTS] (untuk menangani imbalance)")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:12s}: {CLASS_WEIGHTS[i]:.4f}")

In [ ]:
# ==============================================================================
# BAGIAN 13 — ARSITEKTUR AUTOENCODER
# ==============================================================================

def build_autoencoder():
    """
    Convolutional Autoencoder simetris.
    Encoder: 4 blok Conv-BN-ReLU-MaxPool → bottleneck 16×16×256
    Decoder: 4 blok UpSampling-Conv → rekonstruksi 128×128×3
    """
    inp = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS), name='ae_input')

    # ── ENCODER ──────────────────────────────────────────────────────────
    x = tf.keras.layers.Conv2D(32, 3, padding='same', name='enc_conv1a')(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(32, 3, padding='same', name='enc_conv1b')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D(2, name='enc_pool1')(x)              # 64×64

    x = tf.keras.layers.Conv2D(64, 3, padding='same', name='enc_conv2a')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(64, 3, padding='same', name='enc_conv2b')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D(2, name='enc_pool2')(x)              # 32×32

    x = tf.keras.layers.Conv2D(128, 3, padding='same', name='enc_conv3a')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(128, 3, padding='same', name='enc_conv3b')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling2D(2, name='enc_pool3')(x)              # 16×16

    x = tf.keras.layers.Conv2D(256, 3, padding='same', name='enc_conv4a')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(256, 3, padding='same', name='enc_conv4b')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    bottleneck = tf.keras.layers.Activation('relu', name='bottleneck')(x)  # 16×16×256

    # ── DECODER ──────────────────────────────────────────────────────────
    x = tf.keras.layers.UpSampling2D(2)(bottleneck)                        # 32×32
    x = tf.keras.layers.Conv2D(128, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    x = tf.keras.layers.UpSampling2D(2)(x)                                 # 64×64
    x = tf.keras.layers.Conv2D(64, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    x = tf.keras.layers.UpSampling2D(2)(x)                                 # 128×128
    x = tf.keras.layers.Conv2D(32, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    reconstruction = tf.keras.layers.Conv2D(
        CHANNELS, 1, activation='sigmoid', name='reconstruction'
    )(x)

    autoencoder = tf.keras.Model(inp, reconstruction, name='Autoencoder')
    encoder     = tf.keras.Model(inp, bottleneck,     name='Encoder')

    return autoencoder, encoder


def ssim_mse_loss(y_true, y_pred):
    """Loss gabungan: 0.5 * MSE + 0.5 * (1 - SSIM)."""
    mse_loss  = tf.reduce_mean(tf.square(y_true - y_pred))
    ssim_val  = tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    return 0.5 * mse_loss + 0.5 * (1.0 - ssim_val)


autoencoder, encoder = build_autoencoder()
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_AE),
    loss=ssim_mse_loss,
    metrics=['mse']
)

autoencoder.summary()
print(f"\n Autoencoder siap. Parameter: {autoencoder.count_params():,}")
print(f" Encoder siap.     Parameter: {encoder.count_params():,}")

In [ ]:
# ==============================================================================
# BAGIAN 13B — ABLATION STUDY: PENGARUH CLAHE TERHADAP PERFORMA MODEL
# ==============================================================================

ABLATION_SUBSET_SIZE = 1200   # total sampel (stratified) untuk ablation
ABLATION_EPOCHS      = 8
ABLATION_BATCH       = 32

# 1) Ambil subset stratified dari all_records untuk ablation
random.seed(SEED)
_records_by_class = {i: [r for r in all_records if r[1] == i] for i in range(NUM_CLASSES)}
_per_class_n = ABLATION_SUBSET_SIZE // NUM_CLASSES

_ablation_records = []
for i in range(NUM_CLASSES):
    pool = _records_by_class[i]
    _ablation_records += random.sample(pool, min(_per_class_n, len(pool)))
random.shuffle(_ablation_records)

_abl_paths  = [r[0] for r in _ablation_records]
_abl_labels = [r[1] for r in _ablation_records]

_abl_train_p, _abl_test_p, _abl_train_y, _abl_test_y = train_test_split(
    _abl_paths, _abl_labels, test_size=0.20, random_state=SEED, stratify=_abl_labels
)

print(f"[ABLATION] Subset: {len(_abl_paths)} gambar "
      f"(train={len(_abl_train_p)}, test={len(_abl_test_p)})")

# 2) Fungsi preprocessing parametrik (CLAHE on/off + parameter custom)
def preprocess_variant(image: np.ndarray, use_clahe: bool,
                        clip_limit: float = 2.0, tile_grid: tuple = (8, 8)) -> np.ndarray:
    """Preprocessing dengan CLAHE opsional & parameter yang bisa diatur."""
    img = cv2.GaussianBlur(image, (3, 3), 0)
    if use_clahe:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
        l_eq  = clahe.apply(l)
        img   = cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2RGB)
    img = crop_brain_roi(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.astype('float32') / 255.0

def build_dataset_variant(paths, labels, use_clahe, clip_limit=2.0, tile_grid=(8, 8),
                           batch_size=ABLATION_BATCH, training=True):
    """Bangun array numpy hasil preprocessing untuk satu konfigurasi ablation."""
    X = np.zeros((len(paths), IMG_SIZE, IMG_SIZE, CHANNELS), dtype='float32')
    for idx, p in enumerate(paths):
        data = np.fromfile(p, dtype=np.uint8)
        img  = cv2.imdecode(data, cv2.IMREAD_COLOR)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        X[idx] = preprocess_variant(img, use_clahe, clip_limit, tile_grid)
    y = np.array(labels, dtype='int32')
    return X, y

# 3) CNN ringan untuk ablation (bukan arsitektur hybrid penuh)
def build_ablation_cnn():
    model = tf.keras.Sequential([\
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS)),\
        tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),\
        tf.keras.layers.MaxPooling2D(2),\
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),\
        tf.keras.layers.MaxPooling2D(2),\
        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),\
        tf.keras.layers.MaxPooling2D(2),\
        tf.keras.layers.GlobalAveragePooling2D(),\
        tf.keras.layers.Dense(64, activation='relu'),\
        tf.keras.layers.Dropout(0.3),\
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),\
    ], name='AblationCNN')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 4) Jalankan ablation untuk setiap konfigurasi
ablation_configs = [\
    {"name": "Tanpa CLAHE",                      "use_clahe": False, "clip": None, "tile": None},\
    {"name": "CLAHE clip=1.0, tile=(8,8)",       "use_clahe": True,  "clip": 1.0,  "tile": (8, 8)},\
    {"name": "CLAHE clip=2.0, tile=(8,8)",       "use_clahe": True,  "clip": 2.0,  "tile": (8, 8)},\
    {"name": "CLAHE clip=3.0, tile=(8,8)",       "use_clahe": True,  "clip": 3.0,  "tile": (8, 8)},\
    {"name": "CLAHE clip=2.0, tile=(4,4)",       "use_clahe": True,  "clip": 2.0,  "tile": (4, 4)},\
]

ablation_results = []

print("=" * 65)
print("  ABLATION STUDY: PENGARUH CLAHE")
print("=" * 65)

for cfg in ablation_configs:
    print(f"\n[ABLATION] Konfigurasi: {cfg['name']}")

    clip = cfg["clip"] if cfg["clip"] is not None else 2.0
    tile = cfg["tile"] if cfg["tile"] is not None else (8, 8)

    X_train_abl, y_train_abl = build_dataset_variant(
        _abl_train_p, _abl_train_y, cfg["use_clahe"], clip, tile)
    X_test_abl, y_test_abl = build_dataset_variant(
        _abl_test_p, _abl_test_y, cfg["use_clahe"], clip, tile)

    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = build_ablation_cnn()

    es = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)

    hist = model.fit(
        X_train_abl, y_train_abl,
        validation_data=(X_test_abl, y_test_abl),
        epochs=ABLATION_EPOCHS,
        batch_size=ABLATION_BATCH,
        callbacks=[es],
        verbose=0,
    )

    val_loss, val_acc = model.evaluate(X_test_abl, y_test_abl, verbose=0)
    best_epoch = len(hist.history['val_loss']) - 1

    ablation_results.append({
        "Konfigurasi": cfg["name"],
        "Val Accuracy (%)": round(val_acc * 100, 2),
        "Val Loss": round(val_loss, 4),
        "Epoch Terbaik": best_epoch + 1,
    })

    print(f"  -> Val Accuracy: {val_acc*100:.2f}% | Val Loss: {val_loss:.4f} "
          f"(epoch terbaik: {best_epoch + 1})")

# 5) Tabel ringkasan & visualisasi perbandingan
df_ablation = pd.DataFrame(ablation_results)
print("\n" + "=" * 65)
print("  TABEL PERBANDINGAN ABLATION STUDY (CLAHE)")
print("=" * 65)
print(df_ablation.to_string(index=False))

# Simpan tabel ke CSV
_ablation_csv = os.path.join(OUTPUT_DIR, 'ablation_clahe_results.csv')
df_ablation.to_csv(_ablation_csv, index=False)
print(f"\n[DISIMPAN] {_ablation_csv}")

# Visualisasi perbandingan akurasi & loss
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Ablation Study: Pengaruh CLAHE terhadap Performa Model",
              fontweight='bold')

axes[0].barh(df_ablation["Konfigurasi"], df_ablation["Val Accuracy (%)"],
              color='#2ecc71')
axes[0].set_title("Validation Accuracy (%)")
axes[0].set_xlabel("Akurasi (%)")
axes[0].invert_yaxis()

axes[1].barh(df_ablation["Konfigurasi"], df_ablation["Val Loss"],
              color='#e74c3c')
axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Loss")
axes[1].invert_yaxis()

plt.tight_layout()
_p = os.path.join(OUTPUT_DIR, 'ablation_clahe_comparison.png')
plt.savefig(_p, dpi=130, bbox_inches='tight')
plt.show()
print(f"[DISIMPAN] {_p}")

# Tentukan konfigurasi CLAHE terbaik (selain "Tanpa CLAHE")
_clahe_rows = df_ablation[df_ablation["Konfigurasi"] != "Tanpa CLAHE"]
_best_row   = _clahe_rows.loc[_clahe_rows["Val Accuracy (%)"].idxmax()]
print(f"\n[KESIMPULAN] Konfigurasi CLAHE terbaik: {_best_row['Konfigurasi']} "
      f"(Val Accuracy: {_best_row['Val Accuracy (%)']:.2f}%)")

In [ ]:
# ==============================================================================
# BAGIAN 14 — STAGE 1: TRAINING AUTOENCODER
# ==============================================================================

print("=" * 65)
print("  STAGE 1: TRAINING AUTOENCODER")
print("=" * 65)

ae_model_path = os.path.join(OUTPUT_DIR, 'autoencoder_best.keras')

ae_callbacks = [\
    tf.keras.callbacks.ModelCheckpoint(\
        ae_model_path, monitor='val_loss', save_best_only=True,\
        mode='min', verbose=1\
    ),\
    tf.keras.callbacks.EarlyStopping(\
        monitor='val_loss', patience=8, restore_best_weights=True,\
        mode='min', verbose=1\
    ),\
    tf.keras.callbacks.ReduceLROnPlateau(\
        monitor='val_loss', factor=0.5, patience=4,\
        min_lr=1e-6, verbose=1\
    ),\
    tf.keras.callbacks.CSVLogger(\
        os.path.join(OUTPUT_DIR, 'log_stage1_autoencoder.csv')\
    ),\
]

history_ae = autoencoder.fit(
    ae_train_ds,
    validation_data=ae_val_ds,
    epochs=EPOCHS_AE,
    callbacks=ae_callbacks,
    verbose=1,
)

print("\nStage 1 selesai.")

In [ ]:
# ==============================================================================
# BAGIAN 15 — EVALUASI AUTOENCODER (SSIM & MSE)
# ==============================================================================

print("=" * 65)
print("  EVALUASI AUTOENCODER")
print("=" * 65)

# Ambil satu batch untuk evaluasi
eval_batch_imgs = []
for imgs, _ in ae_val_ds.take(4):
    eval_batch_imgs.append(imgs.numpy())
X_sample = np.concatenate(eval_batch_imgs, axis=0)[:100]

reconstructed = autoencoder.predict(X_sample, verbose=0)

mse_vals  = np.mean((X_sample - reconstructed) ** 2, axis=(1, 2, 3))
mean_mse  = float(np.mean(mse_vals))

ssim_vals = []
for orig, recon in zip(X_sample, reconstructed):
    orig_t  = tf.convert_to_tensor(orig[np.newaxis],  dtype=tf.float32)
    recon_t = tf.convert_to_tensor(recon[np.newaxis], dtype=tf.float32)
    ssim    = tf.image.ssim(orig_t, recon_t, max_val=1.0).numpy()[0]
    ssim_vals.append(ssim)
mean_ssim = float(np.mean(ssim_vals))

print(f"  MSE  rata-rata: {mean_mse:.6f}  (target < 0.01)")
print(f"  SSIM rata-rata: {mean_ssim:.4f}  (target > 0.85)")

if mean_ssim >= 0.85 and mean_mse <= 0.01:
    print("  SSIM & MSE memenuhi target. Encoder siap dibekukan.")
elif mean_ssim >= 0.70:
    print("  SSIM cukup baik. Lanjut ke Stage 2.")
else:
    print("  SSIM masih rendah. Pertimbangkan menambah EPOCHS_AE.")

# Visualisasi rekonstruksi
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
fig.suptitle(f'Rekonstruksi Autoencoder (SSIM={mean_ssim:.3f}, MSE={mean_mse:.5f})',
             fontweight='bold')
for j in range(8):
    axes[0, j].imshow(X_sample[j]);         axes[0, j].axis('off')
    axes[1, j].imshow(reconstructed[j]);    axes[1, j].axis('off')
    diff = np.abs(X_sample[j] - reconstructed[j])
    axes[2, j].imshow(diff, cmap='hot');    axes[2, j].axis('off')
    if j == 0:
        axes[0, j].set_title('Original',     fontsize=9)
        axes[1, j].set_title('Rekonstruksi', fontsize=9)
        axes[2, j].set_title('Selisih',      fontsize=9)
plt.tight_layout()
p = os.path.join(OUTPUT_DIR, 'rekonstruksi_autoencoder.png')
plt.savefig(p, dpi=130, bbox_inches='tight'); plt.show()
print(f"[DISIMPAN] {p}")

plt.figure(figsize=(8, 4))
plt.plot(history_ae.history['loss'],     label='Train Loss', linewidth=2)
plt.plot(history_ae.history['val_loss'], label='Val Loss',   linewidth=2, ls='--')
plt.title('Stage 1 — Training Autoencoder', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Loss (MSE+SSIM)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'grafik_stage1_ae.png'), dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# BAGIAN 16 — STAGE 2: MEMBANGUN & TRAINING CLASSIFIER (ENCODER FROZEN)
# ==============================================================================

print("=" * 65)
print("  STAGE 2: TRAINING CLASSIFIER (CUSTOM HYBRID HEAD)")
print("=" * 65)

encoder.trainable = False
print(f"[FREEZE] Encoder dibekukan.")

def build_custom_classifier(encoder_model, num_classes=NUM_CLASSES):
    """
    Pipeline Hybrid Custom Murni:
    Input MRI -> Encoder (frozen) -> Custom Conv Layers -> GAP -> Dense -> Softmax(4)
    """
    inp = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS), name='clf_input')

    latent = encoder_model(inp, training=False)

    x = tf.keras.layers.Conv2D(256, (3, 3), padding='same', activation='relu')(latent)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    x = tf.keras.layers.Conv2D(512, (3, 3), padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.40)(x)

    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.30)(x)

    out = tf.keras.layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = tf.keras.Model(inp, out, name='BrainTumor_Custom_Hybrid_Classifier')
    return model

classifier = build_custom_classifier(encoder)
classifier.summary()

class_weights_dict = {i: float(cw_array[i]) for i in range(NUM_CLASSES)}
print(f"\n Menggunakan Class Weights: {class_weights_dict}")

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_CLF),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

clf_model_path_s2 = os.path.join(OUTPUT_DIR, 'classifier_stage2_best.keras')

clf_callbacks_s2 = [\
    tf.keras.callbacks.ModelCheckpoint(\
        clf_model_path_s2, monitor='val_accuracy',\
        save_best_only=True, mode='max', verbose=1\
    ),\
    tf.keras.callbacks.EarlyStopping(\
        monitor='val_accuracy', patience=7,\
        restore_best_weights=True, mode='max', verbose=1\
    ),\
    tf.keras.callbacks.ReduceLROnPlateau(\
        monitor='val_loss', factor=0.5, patience=3,\
        min_lr=1e-6, verbose=1\
    ),\
    tf.keras.callbacks.CSVLogger(\
        os.path.join(OUTPUT_DIR, 'log_stage2_classifier.csv')\
    ),\
]

history_s2 = classifier.fit(
    clf_train_ds,
    validation_data=clf_val_ds,
    epochs=EPOCHS_CLF,
    callbacks=clf_callbacks_s2,
    class_weight=class_weights_dict,
    verbose=1,
)

print("\n Stage 2 selesai.")

In [ ]:
# ==============================================================================
# BAGIAN 17 — STAGE 3: FINE-TUNING ENCODER & CLASSIFIER
# ==============================================================================

print("=" * 65)
print("  STAGE 3: FINE-TUNING (MEMBUKA KUNCI BLOK TERAKHIR ENCODER)")
print("=" * 65)

classifier.trainable = True

encoder.trainable = True
for layer in encoder.layers:
    if layer.name.startswith('enc_conv4') or layer.name == 'bottleneck':
        layer.trainable = True
    else:
        layer.trainable = False

trainable_count = sum(1 for l in classifier.layers if l.trainable)
print(f"[FINE-TUNE] Membuka blok konvolusi terdalam untuk penyesuaian fitur tumor.")

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FINETUNE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

clf_model_path_s3 = os.path.join(OUTPUT_DIR, 'classifier_stage3_best.keras')

clf_callbacks_s3 = [\
    tf.keras.callbacks.ModelCheckpoint(\
        clf_model_path_s3, monitor='val_accuracy',\
        save_best_only=True, mode='max', verbose=1\
    ),\
    tf.keras.callbacks.EarlyStopping(\
        monitor='val_accuracy', patience=8,\
        restore_best_weights=True, mode='max', verbose=1\
    ),\
    tf.keras.callbacks.ReduceLROnPlateau(\
        monitor='val_loss', factor=0.5, patience=3,\
        min_lr=1e-7, verbose=1\
    ),\
    tf.keras.callbacks.CSVLogger(\
        os.path.join(OUTPUT_DIR, 'log_stage3_finetune.csv')\
    ),\
]

history_s3 = classifier.fit(
    clf_train_ds,
    validation_data=clf_val_ds,
    epochs=EPOCHS_FINETUNE,
    callbacks=clf_callbacks_s3,
    class_weight=CLASS_WEIGHTS,
    verbose=1,
)

print("\n Stage 3 selesai.")

In [ ]:
# ==============================================================================
# BAGIAN 18 — VISUALISASI GRAFIK TRAINING
# ==============================================================================

def plot_combined_history(h2, h3, save_path):
    acc      = h2.history['accuracy']      + h3.history['accuracy']
    val_acc  = h2.history['val_accuracy']  + h3.history['val_accuracy']
    loss     = h2.history['loss']          + h3.history['loss']
    val_loss = h2.history['val_loss']      + h3.history['val_loss']
    ep_s2    = len(h2.history['accuracy'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Grafik Training (Stage 2 + Stage 3 Fine-tuning)', fontweight='bold')

    ax1.plot(acc,     label='Train',      linewidth=2, color='#2196F3')
    ax1.plot(val_acc, label='Validation', linewidth=2, color='#FF9800', ls='--')
    ax1.axvline(ep_s2 - 1, color='red', ls=':', lw=1.5, label='Stage 3 start')
    ax1.set_title('Akurasi'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Akurasi')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(loss,     label='Train',      linewidth=2, color='#2196F3')
    ax2.plot(val_loss, label='Validation', linewidth=2, color='#FF9800', ls='--')
    ax2.axvline(ep_s2 - 1, color='red', ls=':', lw=1.5, label='Stage 3 start')
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()
    print(f"[DISIMPAN] {save_path}")


plot_combined_history(
    history_s2, history_s3,
    os.path.join(OUTPUT_DIR, 'grafik_training_stage2_3.png')
)

In [ ]:
# ==============================================================================
# BAGIAN 19 — EVALUASI INTERNAL (TEST SET)
# ==============================================================================

print("=" * 65)
print("  EVALUASI INTERNAL — TEST SET")
print("=" * 65)

# Kumpulkan prediksi dari test dataset pipeline
y_pred_prob_list, y_true_list = [], []
for imgs, lbls in clf_test_ds:
    y_pred_prob_list.append(classifier(imgs, training=False).numpy())
    y_true_list.append(lbls.numpy())

y_pred_prob = np.concatenate(y_pred_prob_list, axis=0)
y_test_eval = np.concatenate(y_true_list, axis=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
test_acc    = np.mean(y_pred == y_test_eval) * 100

print(f"\nAkurasi Test (Internal): {test_acc:.2f}%")
print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test_eval, y_pred, target_names=CLASSES, digits=4))


def plot_cm(y_true, y_pred, title, save_name):
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontweight='bold')

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
    axes[0].set_title('Jumlah'); axes[0].set_ylabel('Aktual'); axes[0].set_xlabel('Prediksi')

    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
    axes[1].set_title('Normalisasi'); axes[1].set_ylabel('Aktual'); axes[1].set_xlabel('Prediksi')

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, save_name)
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
    print(f"[DISIMPAN] {p}")


def plot_roc(y_true, y_score, title, save_name):
    lb     = LabelBinarizer().fit(range(NUM_CLASSES))
    y_bin  = lb.transform(y_true)
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

    plt.figure(figsize=(8, 6))
    for i, (cls, col) in enumerate(zip(CLASSES, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        plt.plot(fpr, tpr, color=col, lw=2,
                 label=f'{cls} (AUC={auc(fpr, tpr):.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.title(title, fontweight='bold')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right'); plt.grid(alpha=0.3); plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, save_name)
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
    print(f"[DISIMPAN] {p}")


plot_cm(y_test_eval, y_pred, 'Confusion Matrix — Internal Test', 'cm_internal.png')
plot_roc(y_test_eval, y_pred_prob, 'ROC-AUC — Internal Test', 'roc_internal.png')

In [ ]:
# ==============================================================================
# BAGIAN 20 — EXTERNAL VALIDATION (BraTS — Glioma)
# ==============================================================================

print("=" * 65)
print("  EXTERNAL VALIDATION — BRATS (Sisa Subset Uji Murni 80%)")
print("=" * 65)

if len(X_brats_test) > 0:
    y_brats_pred_prob = classifier.predict(X_brats_test, batch_size=BATCH_SIZE, verbose=0)
    y_brats_pred      = np.argmax(y_brats_pred_prob, axis=1)

    # y_brats_test semuanya bernilai 0 (Glioma)
    glioma_acc = np.mean(y_brats_pred == y_brats_test) * 100
    print(f"\nBraTS External Validation:")
    print(f"  Total slice uji murni  : {len(X_brats_test)}")
    print(f"  Diprediksi benar sebagai Glioma: {np.sum(y_brats_pred == 0)} "
          f"({np.mean(y_brats_pred == 0)*100:.1f}%)")
    print(f"  Akurasi generalisasi (glioma): {glioma_acc:.2f}%")

    unique, counts = np.unique(y_brats_pred, return_counts=True)
    print("\n  Distribusi prediksi BraTS:")
    for u, c in zip(unique, counts):
        print(f"    → Kelas '{CLASSES[u]}': {c} ({c/len(X_brats_test)*100:.1f}%)")

    n_show = min(10, len(X_brats_test))
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle(f'BraTS External Validation (Glioma) — Acc: {glioma_acc:.1f}%',
                 fontweight='bold')
    idxs = np.random.choice(len(X_brats_test), n_show, replace=False)
    for i, idx in enumerate(idxs):
        ax       = axes[i // 5, i % 5]
        pred_cls = CLASSES[y_brats_pred[idx]]
        conf     = y_brats_pred_prob[idx].max() * 100
        color    = 'green' if y_brats_pred[idx] == 0 else 'red'
        ax.imshow(X_brats_test[idx])
        ax.set_title(f'{pred_cls}\n{conf:.0f}%', color=color, fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, 'brats_external_validation.png')
    plt.savefig(p, dpi=130, bbox_inches='tight'); plt.show()
    print(f"[DISIMPAN] {p}")

else:
    glioma_acc = None
    print(" BraTS tidak dimuat atau jumlah uji murni kosong.")

In [ ]:
# ==============================================================================
# BAGIAN 20B — RINGKASAN AKURASI INTERNAL vs EKSTERNAL (BraTS)
# ==============================================================================
# Menggabungkan hasil evaluasi internal (Bagian 19: test_acc) dan external
# validation BraTS (Bagian 20: glioma_acc) ke dalam satu ringkasan visual +
# tabel, untuk langsung dipakai pada Bab 4 (Hasil dan Pembahasan).

print("=" * 65)
print("  RINGKASAN AKHIR: AKURASI INTERNAL vs EKSTERNAL")
print("=" * 65)

_summary_rows = [{\
    "Skenario Evaluasi": "Internal (Test Set)",\
    "Jumlah Sampel": len(y_test_eval),\
    "Akurasi (%)": round(float(test_acc), 2),\
}]

if glioma_acc is not None:
    _summary_rows.append({
        "Skenario Evaluasi": "Eksternal (BraTS - Glioma)",
        "Jumlah Sampel": len(X_brats_test),
        "Akurasi (%)": round(float(glioma_acc), 2),
    })

df_summary_acc = pd.DataFrame(_summary_rows)
print(df_summary_acc.to_string(index=False))

_csv_summary = os.path.join(OUTPUT_DIR, 'ringkasan_akurasi_internal_eksternal.csv')
df_summary_acc.to_csv(_csv_summary, index=False)
print(f"\n[DISIMPAN] {_csv_summary}")

fig, ax = plt.subplots(figsize=(6.5, 4.8))
_labels_acc = df_summary_acc["Skenario Evaluasi"].tolist()
_vals_acc   = df_summary_acc["Akurasi (%)"].tolist()
_colors_acc = ['#2ecc71', '#3498db'][:len(_vals_acc)]

bars = ax.bar(_labels_acc, _vals_acc, color=_colors_acc)
for b, v in zip(bars, _vals_acc):
    ax.text(b.get_x() + b.get_width()/2, v + 1, f"{v:.2f}%",
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Akurasi (%)')
ax.set_ylim(0, 105)
ax.set_title('Perbandingan Akurasi Internal (Test Set) vs Eksternal (BraTS)',
              fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
_p_acc = os.path.join(OUTPUT_DIR, 'ringkasan_akurasi_internal_eksternal.png')
plt.savefig(_p_acc, dpi=130, bbox_inches='tight')
plt.show()
print(f"[DISIMPAN] {_p_acc}")

In [ ]:
# ==============================================================================
# BAGIAN 21 — GRAD-CAM VISUALIZATION
# ==============================================================================

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array[np.newaxis], tf.float32)
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads        = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
    )
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    img_uint8 = np.uint8(img * 255)
    return np.uint8(img_uint8 * (1 - alpha) + heatmap_colored * alpha)


last_conv_name = None
for layer in reversed(classifier.layers):
    if isinstance(layer, tf.keras.layers.Activation):
        last_conv_name = layer.name
        break
if last_conv_name is None:
    for layer in reversed(classifier.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_name = layer.name
            break

print(f"[GRADCAM] Menggunakan layer: {last_conv_name}")

# Kumpulkan sedikit sampel test untuk visualisasi GradCAM
X_test_sample, y_test_sample = [], []
for imgs, lbls in clf_test_ds.take(4):
    X_test_sample.append(imgs.numpy())
    y_test_sample.append(lbls.numpy())
X_test_sample = np.concatenate(X_test_sample)
y_test_sample = np.concatenate(y_test_sample)

fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(14, 4 * NUM_CLASSES))
fig.suptitle('Grad-CAM Visualization (Perhatian Model pada Area Tumor)',
             fontweight='bold', fontsize=13)

for i, cls in enumerate(CLASSES):
    cls_idxs = np.where(y_test_sample == i)[0]
    chosen   = np.random.choice(cls_idxs, min(2, len(cls_idxs)), replace=False)
    for j, idx in enumerate(chosen):
        col = j * 2
        img = X_test_sample[idx]
        try:
            heatmap = make_gradcam_heatmap(img, classifier, last_conv_name, pred_index=i)
            overlay = overlay_gradcam(img, heatmap)
            axes[i, col].imshow(img)
            axes[i, col].set_title(f'{cls}\nOriginal', fontsize=8)
            axes[i, col].axis('off')
            axes[i, col + 1].imshow(overlay)
            axes[i, col + 1].set_title(f'{cls}\nGrad-CAM', fontsize=8)
            axes[i, col + 1].axis('off')
        except Exception:
            axes[i, col].axis('off')
            axes[i, col + 1].axis('off')

plt.tight_layout()
p = os.path.join(OUTPUT_DIR, 'gradcam_visualization.png')
plt.savefig(p, dpi=130, bbox_inches='tight'); plt.show()
print(f"[DISIMPAN] {p}")

In [ ]:
# ==============================================================================
# BAGIAN 22 — SIMPAN MODEL & RINGKASAN AKHIR
# ==============================================================================

autoencoder.save(os.path.join(OUTPUT_DIR, 'autoencoder_final.keras'))
encoder.save(os.path.join(OUTPUT_DIR, 'encoder_final.keras'))
classifier.save(os.path.join(OUTPUT_DIR, 'classifier_final.keras'))
classifier.save(os.path.join(OUTPUT_DIR, 'classifier_final.h5'))

print("  RINGKASAN HASIL AKHIR ")
print(f"  SSIM Autoencoder        : {mean_ssim:.4f}  (target > 0.85)")
print(f"  MSE Autoencoder         : {mean_mse:.6f} (target < 0.01)")
print(f"  Akurasi Test (Internal) : {test_acc:.2f}%")

if glioma_acc is not None:
    print(f"  Akurasi External (BraTS): {glioma_acc:.2f}% (Glioma only)")

print(f"\n  File output di: {OUTPUT_DIR}/")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f"    {f:45s} ({size:8.1f} KB)")

print("  Selesai!")